# Feature Engineering: Target Encoding cho cặp Store-Family

Notebook này tạo ra feature **historical average sales** cho mỗi cặp `store_nbr`–`family` bằng phương pháp **Target Encoding**.
Sử dụng thư viện `category_encoders` với tham số `smoothing` để xử lý các cặp có ít dữ liệu lịch sử.

## Feature được tạo ra
| Feature | Mô tả |
|---|---|
| `store_family_te` | Target-encoded mean sales cho mỗi cặp store-family, được tính bằng KFold out-of-fold để chống data leakage |


```
Load Data → Build store_family key → KFold OOF Encoding (leakage-safe)
    → Full-train Encoding for Test → Save Output
```

## 1. Import Thư viện

- **pandas**: đọc và xử lý dữ liệu dạng bảng.
- **numpy**: mảng số (dùng cho `oof` array).
- **KFold**: chia tập train thành các fold để thực hiện out-of-fold encoding.
- **TargetEncoder**: encoder từ `category_encoders` hỗ trợ smoothing cho các nhóm ít dữ liệu.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from category_encoders import TargetEncoder

## 2. Đọc Dữ liệu

Pipeline cần hai bộ dữ liệu:
- **`train_cleaned.csv`** — tập huấn luyện chứa cột `sales` (target) cùng `store_nbr` và `family`.
- **`test_cleaned.csv`** — tập kiểm tra cần được encode bằng thống kê từ tập train.

In [2]:
BASE_PATH = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data'

df_train = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')
df_test  = pd.read_csv(rf'{BASE_PATH}\processed\test_cleaned.csv')

print(f'Train shape : {df_train.shape}')
print(f'Test shape  : {df_test.shape}')
df_train.head(3)

Train shape : (3000888, 6)
Test shape  : (28512, 5)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0


In [3]:
# Local aliases used by the explanatory cells below
# These point to the loaded dataframes so cells that reference
# `train` / `test` / `target` work when running the notebook.
train = df_train.copy()
test = df_test.copy()
target = 'sales'
# Ensure composite key exists so explanatory cells can run independently
train['store_family'] = train['store_nbr'].astype(str) + '_' + train['family']
test['store_family']  = test['store_nbr'].astype(str)  + '_' + test['family']
print('Aliases set: train/test/target and store_family created')

Aliases set: train/test/target and store_family created


---

## Phần 1 — Tại sao cần Target Encoding? Tại sao cần chống Leakage?

### Target Encoding là gì?

Target Encoding thay thế một biến phân loại (categorical) bằng **giá trị trung bình của target** (ở đây là `sales`) cho mỗi nhóm. Ví dụ, cặp `store_nbr=1, family=AUTOMOTIVE` sẽ được thay bằng mean sales lịch sử của chính cặp đó.

**Ưu điểm** so với One-Hot Encoding:
- Không tạo ra hàng nghìn cột (54 stores x 33 families = 1782 cặp).
- Mang thông tin trực tiếp về mối quan hệ giữa feature và target.

### Tại sao tính mean trên toàn bộ dataset là Data Leakage?

Giả sử ta tính `mean_sales` cho cặp `(store=1, AUTOMOTIVE)` trên **toàn bộ** tập train:

```
mean_sales = train[train.store_family == '1_AUTOMOTIVE']['sales'].mean()
```

Khi đó, **mỗi dòng trong nhóm đó đã đóng góp vào chính giá trị mean mà nó nhận được**. Điều này có nghĩa:
- Mô hình "nhìn thấy" target của chính observation đó (dưới dạng trung bình).
- Kết quả validation bị lạc quan (overly optimistic) — mô hình có vẻ tốt trên train/val nhưng kém trên dữ liệu mới.

### Giải pháp: KFold Out-of-Fold Encoding

Chia tập train thành K fold. Với mỗi fold:
1. **Fit** encoder trên K-1 fold (không chứa observation đang cần encode).
2. **Transform** fold còn lại — observation **không bao giờ** tham gia vào phép tính mean của chính mình.

Kết quả: mỗi dòng trong train nhận được một giá trị target-encoded **không bị nhiễm** bởi chính nó.

## Phần 2 — Hàm `add_target_encoding` — Toàn bộ Pipeline

Hàm dưới đây chứa toàn bộ logic target encoding. Code được giữ nguyên từ file gốc `feature_target_encoding.py`.

In [4]:
def add_target_encoding(train, test, target="sales", n_splits=5):

    train, test = train.copy(), test.copy()

    # Combine key
    train["store_family"] = (
        train["store_nbr"].astype(str) + "_" + train["family"]
    )
    test["store_family"] = (
        test["store_nbr"].astype(str) + "_" + test["family"]
    )

    kf = KFold(n_splits=n_splits, shuffle=False)
    oof = np.zeros(len(train))

    # ---- Out-of-fold encoding (leakage safe) ----
    for tr_idx, val_idx in kf.split(train):

        enc = TargetEncoder(cols=["store_family"], smoothing=10)

        # Fit ONLY on training fold
        enc.fit(
            train.iloc[tr_idx][["store_family"]],
            train.iloc[tr_idx][target]
        )

        # Transform validation fold (never sees its own target)
        oof[val_idx] = enc.transform(
            train.iloc[val_idx][["store_family"]]
        )["store_family"]

    train["store_family_te"] = oof

    # ---- Fit on full train for test ----
    final_enc = TargetEncoder(cols=["store_family"], smoothing=10)
    final_enc.fit(train[["store_family"]], train[target])

    # Test uses only train statistics
    test["store_family_te"] = final_enc.transform(
        test[["store_family"]]
    )["store_family"]

    return train, test

## Phần 3 — Giải thích từng bước chống Data Leakage

Dưới đây là phân tích chi tiết **tại sao** mỗi bước trong hàm `add_target_encoding` đảm bảo không xảy ra data leakage.

### Bước 1 — Tạo composite key `store_family`

Ghép `store_nbr` và `family` thành một chuỗi duy nhất (ví dụ: `"1_AUTOMOTIVE"`).

**Về leakage**: Bước này **không liên quan** đến target — chỉ tạo identifier từ các feature có sẵn, nên hoàn toàn an toàn.

**Tại sao cần composite key?** Vì ta muốn tính mean sales cho **từng cặp** store-family, không phải chỉ theo store hoặc chỉ theo family. Mỗi cửa hàng bán các mặt hàng khác nhau với mức doanh số khác nhau.

In [5]:
train["store_family"] = train["store_nbr"].astype(str) + "_" + train["family"]

### Bước 2 — KFold Out-of-Fold Encoding (core leakage protection)

Đây là bước **quan trọng nhất** để chống data leakage.

**Tại sao chống được leakage?**

1. **`enc.fit()` chỉ nhận `tr_idx`** — encoder tính mean sales **chỉ từ 4/5 dữ liệu** (training fold). Các observation trong `val_idx` **không tham gia** vào phép tính này.

2. **`enc.transform()` chỉ áp dụng cho `val_idx`** — mỗi observation nhận giá trị mean được tính từ dữ liệu **không chứa chính nó**.

3. **Encoder mới cho mỗi iteration** — `enc = TargetEncoder(...)` được khởi tạo lại trong mỗi vòng lặp, đảm bảo không có thông tin "rò rỉ" giữa các fold.

4. **`smoothing=10`** — Tham số smoothing pha trộn mean của nhóm nhỏ với global mean, giảm overfitting cho các cặp store-family có ít observation. Công thức: `encoded = (count * group_mean + smoothing * global_mean) / (count + smoothing)`.

5. **`shuffle=False`** — Giữ thứ tự thời gian nguyên vẹn, phù hợp với bài toán time series (tránh trộn lẫn future data vào past folds).

In [6]:
kf = KFold(n_splits=5, shuffle=False)
oof = np.zeros(len(train))

for tr_idx, val_idx in kf.split(train):
    enc = TargetEncoder(cols=["store_family"], smoothing=10)
    enc.fit(train.iloc[tr_idx][["store_family"]], train.iloc[tr_idx][target])
    oof[val_idx] = enc.transform(train.iloc[val_idx][["store_family"]])["store_family"]

### Bước 3 — Full-train encoding cho Test Set

**Tại sao bước này an toàn?**

- Encoder được fit trên **toàn bộ tập train** — đây là hợp lệ vì test set **không bao giờ** tham gia vào quá trình fit.
- Test set chỉ **nhận** giá trị transform từ thống kê đã tính trên train — không có thông tin nào từ test "rò rỉ" ngược lại.
- Dùng toàn bộ train (thay vì chỉ 4/5) giúp encoder có thêm dữ liệu, cho ước lượng mean chính xác hơn.

**Tại sao không dùng KFold cho test?** Vì test set không có target (`sales`), nên không thể fit encoder trên test. Ta chỉ cần transform test bằng encoder đã fit trên train.

In [7]:
final_enc = TargetEncoder(cols=["store_family"], smoothing=10)
final_enc.fit(train[["store_family"]], train[target])
test["store_family_te"] = final_enc.transform(test[["store_family"]])["store_family"]

## Phần 4 — Danh sách tên Feature

Khai báo tên feature dưới dạng constant để các notebook downstream dễ dàng import và sử dụng.

In [8]:
TARGET_ENCODING_FEATURES = [
    'store_family_te',
]

print(f'Target encoding features ({len(TARGET_ENCODING_FEATURES)}):')
for f in TARGET_ENCODING_FEATURES:
    print(f'  - {f}')

Target encoding features (1):
  - store_family_te


## Phần 5 — Chạy Pipeline

Áp dụng hàm `add_target_encoding` lên dữ liệu thực. Kiểm tra kết quả bằng thống kê tổng quan và một vài dòng mẫu.

In [9]:
df_train_encoded, df_test_encoded = add_target_encoding(df_train, df_test)

print('=== Train — Target Encoding Feature ===')
print(df_train_encoded['store_family_te'].describe())
print()
print('=== Test — Target Encoding Feature ===')
print(df_test_encoded['store_family_te'].describe())
print()
print('=== Sample rows (Train) ===')
df_train_encoded[['store_nbr', 'family', 'store_family', 'sales', 'store_family_te']].head(10)

=== Train — Target Encoding Feature ===
count    3.000888e+06
mean     3.577729e+02
std      9.649663e+02
min      0.000000e+00
25%      2.691908e+00
50%      1.942168e+01
75%      2.225125e+02
max      1.025046e+04
Name: store_family_te, dtype: float64

=== Test — Target Encoding Feature ===
count    28512.000000
mean       357.775749
std        961.600889
min          0.000000
25%          2.877078
50%         19.812350
75%        221.086105
max       9730.436483
Name: store_family_te, dtype: float64

=== Sample rows (Train) ===


,store_nbr,family,store_family,sales,store_family_te
0,1,AUTOMOTIVE,1_AUTOMOTIVE,0.0,3.523385
1,1,BABY CARE,1_BABY CARE,0.0,0.000000
2,1,BEAUTY,1_BEAUTY,0.0,2.561247
3,1,BEVERAGES,1_BEVERAGES,0.0,1772.723088
4,1,BOOKS,1_BOOKS,0.0,0.156644
5,1,BREAD/BAKERY,1_BREAD/BAKERY,0.0,355.752459
6,1,CELEBRATION,1_CELEBRATION,0.0,12.461767
7,1,CLEANING,1_CLEANING,0.0,650.435783
8,1,DAIRY,1_DAIRY,0.0,680.178916
9,1,DELI,1_DELI,0.0,126.689168


## Lưu Kết quả

Lưu dataframe đã được bổ sung feature target encoding vào thư mục processed để các notebook modelling downstream có thể sử dụng.

In [10]:
# NOTE: CSV writes removed — train_target_encoding.csv and test_target_encoding.csv
# are no longer saved separately. Target encoding (store_family_te) is computed
# inline in build_train_final.py and merged directly into train_final.csv.
#
# TRAIN_OUTPUT = rf'{BASE_PATH}\processed	rain_target_encoding.csv'
# TEST_OUTPUT  = rf'{BASE_PATH}\processed	est_target_encoding.csv'
# df_train_encoded.to_csv(TRAIN_OUTPUT, index=False)
# df_test_encoded.to_csv(TEST_OUTPUT, index=False)
print('[SKIPPED] CSV writes disabled — target encoding merged in build_train_final.py')
print(f'Train encoded shape : {df_train_encoded.shape}')
print(f'Test encoded shape  : {df_test_encoded.shape}')


Train saved to : D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\train_target_encoding.csv
Train shape    : (3000888, 8)
Test saved to  : D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\test_target_encoding.csv
Test shape     : (28512, 7)
